# ZeroMRG COV-CTR 0% paired: resume text training from epoch 95

This notebook resumes the **text-autoencoder decoder stage** from the verified epoch-95 checkpoint. It does not start image-conditioned zero-shot inference yet. The paper trains the decoder from frozen, L2-normalized M-CLIP report embeddings using SGD (`lr=1e-3`, `momentum=0.99`) and early stopping after 50 validation epochs without improvement. The paper does not specify a scheduler or a maximum epoch count; this reproduction therefore retains **no scheduler** and uses 200 only as a safety ceiling.

Important: `/kaggle/input` is read-only and `/kaggle/working` is temporary. This notebook never modifies the uploaded epoch-95 artifact. It writes new continuation checkpoints under a separate working directory. Download the generated backup ZIP or use **Save Version** before the session expires.

## 1. Kaggle setup

Enable a GPU accelerator and Internet in Notebook settings. Attach the Kaggle dataset containing `zeroMRGC_epoch95.zip`, or the already-expanded epoch-95 files. The package install message saying that a restart *may* be needed is generic; the import test below is authoritative.

In [ ]:
%pip -q install "pycocoevalcap>=1.2,<2"

import gc
import hashlib
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('pycocoevalcap import: OK')

In [ ]:
# Reproduction constants. Do not change these while continuing this checkpoint.
SEED = 42
EXPECTED_SOURCE_EPOCH = 95
MAX_EPOCHS = 200
PATIENCE = 50
BATCH_SIZE = 6
LEARNING_RATE = 1e-3
MOMENTUM = 0.99
D_MODEL = 768
GRADIENT_CLIP = 5.0
BACKUP_EVERY = 10

# Leave blank for automatic discovery. If Kaggle finds several candidate datasets,
# set this to the exact ZIP or directory path under /kaggle/input.
SOURCE_ARTIFACT = ''
INPUT_ROOT = Path('/kaggle/input')
RUN_ROOT = Path('/kaggle/working/zeromrg_covctr_resume_epoch095')
RESTORED_ROOT = RUN_ROOT / 'restored_epoch095'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
RESULTS_DIR = RUN_ROOT / 'results'
for directory in (RESTORED_ROOT, CHECKPOINT_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Enable a Kaggle GPU accelerator, then restart this session.'
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed_all(SEED)

gpu = torch.cuda.get_device_properties(0)
print('Device:', DEVICE)
print('GPU:', gpu.name)
print('GPU memory (GB):', round(gpu.total_memory / 1024**3, 2))
print('Run output:', RUN_ROOT)
print('Epoch ceiling:', MAX_EPOCHS, '| patience:', PATIENCE)

## 2. Restore the immutable epoch-95 artifact

The cell accepts either a ZIP or an expanded Kaggle dataset. It protects against path traversal, never writes into `/kaggle/input`, and is safe to rerun. If automatic discovery reports multiple candidates, set `SOURCE_ARTIFACT` in the previous cell.

In [ ]:
REQUIRED_NAMES = {
    'text_decoder_last.pt', 'text_decoder_best.pt',
    'train_manifest.csv', 'validation_manifest.csv', 'test_manifest.csv',
    'prompt_manifest.csv', 'paired_10_manifest.csv',
    'decoder_vocabulary.json', 'mclip_text_features.pt',
    'mclip_prompt_bank.pt', 'text_training_history.json',
}

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)

def choose_source():
    if SOURCE_ARTIFACT:
        source = Path(SOURCE_ARTIFACT)
        assert source.exists(), f'SOURCE_ARTIFACT does not exist: {source}'
        return source
    direct = list(INPUT_ROOT.rglob('text_decoder_last.pt'))
    complete_direct = [p.parent for p in direct if (p.parent / 'decoder_vocabulary.json').exists()]
    preferred_direct = [p for p in complete_direct if re.search(r'epoch[_-]?0*95|backup5', str(p), re.I)]
    direct_pool = preferred_direct or complete_direct
    if len(direct_pool) == 1:
        return direct_pool[0]
    zips = list(INPUT_ROOT.rglob('*.zip'))
    preferred_zips = [p for p in zips if re.search(r'epoch[_-]?0*95|zeromrgc.*95|backup5', p.name, re.I)]
    zip_pool = preferred_zips or zips
    if len(zip_pool) == 1:
        return zip_pool[0]
    print('Expanded candidates:', *complete_direct, sep='\n  ')
    print('ZIP candidates:', *zips, sep='\n  ')
    raise RuntimeError('Could not choose exactly one source. Set SOURCE_ARTIFACT explicitly.')

source = choose_source()
print('Selected source:', source)
if source.is_file():
    print('Source ZIP SHA-256:', sha256(source))
    if not (RESTORED_ROOT / 'text_decoder_last.pt').exists():
        safe_extract(source, RESTORED_ROOT)
else:
    for path in source.iterdir():
        if path.is_file() and not (RESTORED_ROOT / path.name).exists():
            shutil.copy2(path, RESTORED_ROOT / path.name)

last_candidates = list(RESTORED_ROOT.rglob('text_decoder_last.pt'))
assert len(last_candidates) == 1, f'Expected one restored last checkpoint, found {last_candidates}'
ARTIFACT_DIR = last_candidates[0].parent
missing = sorted(name for name in REQUIRED_NAMES if not (ARTIFACT_DIR / name).exists())
assert not missing, f'Restored artifact is incomplete; missing: {missing}'
print('Artifact directory:', ARTIFACT_DIR)
print('All required epoch-95 files are available.')

In [ ]:
source_checkpoint = torch.load(
    ARTIFACT_DIR / 'text_decoder_last.pt', map_location='cpu', weights_only=False
)
required_checkpoint_keys = {
    'epoch', 'decoder', 'optimizer', 'best_validation_bleu4',
    'epochs_without_improvement', 'history',
}
missing_keys = sorted(required_checkpoint_keys - set(source_checkpoint))
assert not missing_keys, f'Checkpoint keys missing: {missing_keys}'
assert source_checkpoint['epoch'] >= EXPECTED_SOURCE_EPOCH, (
    f"Expected epoch {EXPECTED_SOURCE_EPOCH} or newer, got {source_checkpoint['epoch']}"
)
assert len(source_checkpoint['history']) == EXPECTED_SOURCE_EPOCH
assert source_checkpoint['history'][-1]['epoch'] == EXPECTED_SOURCE_EPOCH
scheduler_state = source_checkpoint.get('scheduler')
assert scheduler_state in (None, {}), (
    'This checkpoint contains an unknown scheduler state. Do not silently discard it; '
    'restore the exact scheduler implementation before training.'
)
print('Verified source epoch:', source_checkpoint['epoch'])
print('Best validation BLEU-4:', source_checkpoint['best_validation_bleu4'])
print('Patience:', source_checkpoint['epochs_without_improvement'], '/', PATIENCE)
print('History entries:', len(source_checkpoint['history']))
print('Scheduler: none (consistent with the paper and original run)')
print('Last checkpoint SHA-256:', sha256(ARTIFACT_DIR / 'text_decoder_last.pt'))
del source_checkpoint
gc.collect()

## 3. Reload the exact partitions, vocabulary, and cached M-CLIP features

No report cleaning, splitting, tokenization, or embedding recomputation occurs here. The saved artifacts ensure that continuation uses the same 571/71/72 split, 368-token vocabulary, and normalized text conditions used through epoch 95.

In [ ]:
def read_manifest(name):
    return pd.read_csv(ARTIFACT_DIR / name, dtype={'image_id': str})

train_df = read_manifest('train_manifest.csv')
validation_df = read_manifest('validation_manifest.csv')
test_df = read_manifest('test_manifest.csv')
prompt_df = read_manifest('prompt_manifest.csv')
paired_10_df = read_manifest('paired_10_manifest.csv')
REPORT_COLUMN = 'report' if 'report' in train_df.columns else 'reports_En'
assert REPORT_COLUMN in train_df.columns

with open(ARTIFACT_DIR / 'decoder_vocabulary.json', encoding='utf-8') as handle:
    vocabulary_artifact = json.load(handle)
vocabulary = vocabulary_artifact['vocabulary']
token_to_id = {token: index for index, token in enumerate(vocabulary)}
PAD_ID = vocabulary_artifact['special_token_ids']['pad']
BOS_ID = vocabulary_artifact['special_token_ids']['bos']
EOS_ID = vocabulary_artifact['special_token_ids']['eos']
UNK_ID = vocabulary_artifact['special_token_ids']['unk']
MAX_TOKENS = vocabulary_artifact['max_tokens']
TOKEN_RE = re.compile(r'\w+|[^\w\s]', re.UNICODE)

def tokenize_report(text):
    return TOKEN_RE.findall(str(text))

def encode_report(text):
    ids = [BOS_ID] + [token_to_id.get(t, UNK_ID) for t in tokenize_report(text)] + [EOS_ID]
    if len(ids) > MAX_TOKENS:
        raise ValueError(f'Report length {len(ids)} exceeds MAX_TOKENS={MAX_TOKENS}')
    return ids

def decode_report(token_ids):
    tokens = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == EOS_ID:
            break
        if token_id not in {PAD_ID, BOS_ID}:
            tokens.append(vocabulary[token_id])
    text = ' '.join(tokens)
    text = re.sub(r'\s+([.,!?;:%)\]])', r'\1', text)
    return re.sub(r'([(\[])\s+', r'\1', text).strip()

text_cache = torch.load(ARTIFACT_DIR / 'mclip_text_features.pt', map_location='cpu', weights_only=False)
train_text_features = text_cache['train']
validation_text_features = text_cache['validation']
prompt_cache = torch.load(ARTIFACT_DIR / 'mclip_prompt_bank.pt', map_location='cpu', weights_only=False)
prompt_embeddings_cpu = prompt_cache['embeddings']

assert (len(train_df), len(validation_df), len(test_df)) == (571, 71, 72)
assert len(prompt_df) == 500 and len(paired_10_df) == 57
assert len(vocabulary) == 368 and MAX_TOKENS == 160
assert len(train_text_features) == 571 and len(validation_text_features) == 71
assert tuple(prompt_embeddings_cpu.shape) == (500, 768)
print('Train:', len(train_df), '| validation:', len(validation_df), '| test:', len(test_df))
print('Prompts:', len(prompt_df), '| vocabulary:', len(vocabulary), '| max tokens:', MAX_TOKENS)
print('Cached text conditions:', len(train_text_features), len(validation_text_features))

In [ ]:
class ReportDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        report = row[REPORT_COLUMN]
        return {
            'row_index': index,
            'image_id': str(row['image_id']),
            'report': report,
            'token_ids': torch.tensor(encode_report(report), dtype=torch.long),
        }

def collate_reports(examples):
    width = max(len(item['token_ids']) for item in examples)
    ids = torch.full((len(examples), width), PAD_ID, dtype=torch.long)
    for row, item in enumerate(examples):
        ids[row, :len(item['token_ids'])] = item['token_ids']
    return {
        'row_index': [item['row_index'] for item in examples],
        'image_id': [item['image_id'] for item in examples],
        'report': [item['report'] for item in examples],
        'token_ids': ids,
    }

def make_loader(frame, shuffle=False, epoch=None):
    generator = None
    if shuffle:
        generator = torch.Generator().manual_seed(SEED + int(epoch))
    return DataLoader(
        ReportDataset(frame), batch_size=BATCH_SIZE, shuffle=shuffle,
        generator=generator, num_workers=0, collate_fn=collate_reports,
    )

def feature_batch(cache, batch):
    if isinstance(cache, dict):
        values = [cache[key] for key in batch['image_id']]
        return torch.stack([torch.as_tensor(value) for value in values]).to(DEVICE, dtype=torch.float32)
    indices = torch.tensor(batch['row_index'], dtype=torch.long)
    return torch.as_tensor(cache)[indices].to(DEVICE, dtype=torch.float32)

probe_batch = next(iter(make_loader(train_df)))
print('Batch token shape:', tuple(probe_batch['token_ids'].shape))
print('Condition shape:', tuple(feature_batch(train_text_features, probe_batch).shape))

## 4. Recreate the checkpoint-compatible decoder

This is the exact architecture used to produce the uploaded state: 768-dimensional token/position embeddings, six Transformer decoder layers, eight attention heads, 3072-dimensional feed-forward blocks, GELU, and dropout 0.1. It has 57,399,152 parameters for this vocabulary. Do not substitute the repository's newer modular decoder while resuming this checkpoint because its state-dict naming and forward details differ.

In [ ]:
class ReportDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(len(vocabulary), D_MODEL, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(MAX_TOKENS, D_MODEL)
        layer = nn.TransformerDecoderLayer(
            d_model=D_MODEL, nhead=8, dim_feedforward=3072, dropout=0.1,
            activation='gelu', batch_first=True,
        )
        self.transformer = nn.TransformerDecoder(layer, num_layers=6)
        self.output_projection = nn.Linear(D_MODEL, len(vocabulary))

    def forward(self, condition, input_ids):
        batch_size, sequence_length = input_ids.shape
        if sequence_length > MAX_TOKENS:
            raise ValueError(f'Sequence length {sequence_length} exceeds {MAX_TOKENS}')
        if tuple(condition.shape) != (batch_size, D_MODEL):
            raise ValueError(f'Invalid condition shape: {tuple(condition.shape)}')
        positions = torch.arange(sequence_length, device=input_ids.device).unsqueeze(0)
        target = self.token_embedding(input_ids) + self.position_embedding(positions)
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=input_ids.device),
            diagonal=1,
        )
        memory = condition.unsqueeze(1).to(dtype=target.dtype)
        hidden = self.transformer(
            tgt=target, memory=memory, tgt_mask=causal_mask,
            tgt_key_padding_mask=input_ids.eq(PAD_ID),
        )
        return self.output_projection(hidden)

    @torch.no_grad()
    def generate(self, condition, max_length=MAX_TOKENS):
        self.eval()
        generated = torch.full(
            (condition.size(0), 1), BOS_ID, dtype=torch.long, device=condition.device
        )
        finished = torch.zeros(condition.size(0), dtype=torch.bool, device=condition.device)
        for _ in range(max_length - 1):
            next_token = self(condition, generated)[:, -1].argmax(dim=-1)
            next_token = torch.where(finished, torch.full_like(next_token, PAD_ID), next_token)
            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished |= next_token.eq(EOS_ID)
            if bool(finished.all()):
                break
        return generated

decoder = ReportDecoder().to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in decoder.parameters())
assert parameter_count == 57_399_152, parameter_count
assert all(parameter.requires_grad for parameter in decoder.parameters())
print('Decoder parameters:', parameter_count)
print('Trainable component: decoder only')
print('M-CLIP is not loaded; its frozen, normalized features are cached on CPU.')

In [ ]:
ptb = PTBTokenizer()
assert shutil.which('java'), 'Kaggle Java is required by the COCO PTB tokenizer.'

def calculate_decoder_loss(conditions, complete_ids):
    logits = decoder(conditions, complete_ids[:, :-1])
    targets = complete_ids[:, 1:]
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
        ignore_index=PAD_ID, reduction='mean',
    )

def bleu4_score(references, predictions):
    refs = {i: [{'caption': text}] for i, text in enumerate(references)}
    preds = {i: [{'caption': text}] for i, text in enumerate(predictions)}
    tokenized_refs = ptb.tokenize(refs)
    tokenized_preds = ptb.tokenize(preds)
    scores, _ = Bleu(4).compute_score(tokenized_refs, tokenized_preds)
    return 100.0 * float(scores[3])

@torch.no_grad()
def validate_text_autoencoder():
    decoder.eval()
    references, predictions = [], []
    for batch in make_loader(validation_df):
        conditions = feature_batch(validation_text_features, batch)
        generated = decoder.generate(conditions).cpu()
        references.extend(batch['report'])
        predictions.extend(decode_report(row) for row in generated)
    return bleu4_score(references, predictions), references, predictions

## 5. Restore the newest valid training state and run a non-mutating preflight

On the first run, the source is epoch 95. If this notebook is rerun in the same Kaggle version/session after making progress, it automatically prefers the newer continuation checkpoint. The optimizer is restored, including SGD momentum buffers. No optimizer step is taken during preflight.

In [ ]:
SOURCE_LAST = ARTIFACT_DIR / 'text_decoder_last.pt'
SOURCE_BEST = ARTIFACT_DIR / 'text_decoder_best.pt'
RESUME_LAST = CHECKPOINT_DIR / 'text_decoder_last_resume.pt'
SELECTED_BEST = CHECKPOINT_DIR / 'text_decoder_best_selected.pt'
HISTORY_JSON = RESULTS_DIR / 'text_training_history_resume.json'
HISTORY_CSV = RESULTS_DIR / 'text_training_history_resume.csv'
TRAINING_LOG = RESULTS_DIR / 'text_training_resume.log'

if not SELECTED_BEST.exists():
    shutil.copy2(SOURCE_BEST, SELECTED_BEST)

def checkpoint_epoch(path):
    if not path.exists():
        return -1
    value = torch.load(path, map_location='cpu', weights_only=False)
    return int(value['epoch'])

resume_from = max([SOURCE_LAST, RESUME_LAST], key=checkpoint_epoch)
checkpoint = torch.load(resume_from, map_location=DEVICE, weights_only=False)
assert checkpoint.get('scheduler') in (None, {})

model_keys = decoder.state_dict()
saved_keys = checkpoint['decoder']
assert set(model_keys) == set(saved_keys), (
    f'Model/checkpoint keys differ. Missing={set(model_keys)-set(saved_keys)}, '
    f'unexpected={set(saved_keys)-set(model_keys)}'
)
shape_errors = {k: (tuple(model_keys[k].shape), tuple(saved_keys[k].shape))
                for k in model_keys if model_keys[k].shape != saved_keys[k].shape}
assert not shape_errors, f'Checkpoint shape mismatch: {shape_errors}'
decoder.load_state_dict(checkpoint['decoder'], strict=True)

optimizer = torch.optim.SGD(decoder.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
optimizer.load_state_dict(checkpoint['optimizer'])
for state in optimizer.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(DEVICE)

start_epoch = int(checkpoint['epoch']) + 1
best_validation_bleu4 = float(checkpoint['best_validation_bleu4'])
epochs_without_improvement = int(checkpoint['epochs_without_improvement'])
training_history = list(checkpoint['history'])
assert start_epoch >= EXPECTED_SOURCE_EPOCH + 1
assert training_history[-1]['epoch'] == start_epoch - 1
momentum_buffers = sum('momentum_buffer' in state for state in optimizer.state.values())
assert momentum_buffers > 0, 'SGD momentum buffers were not restored.'

decoder.eval()
with torch.no_grad():
    conditions = feature_batch(train_text_features, probe_batch)
    ids = probe_batch['token_ids'].to(DEVICE)
    preflight_loss = calculate_decoder_loss(conditions, ids)
assert tuple(conditions.shape) == (len(probe_batch['image_id']), 768)
assert bool(torch.isfinite(preflight_loss))
assert all(parameter.grad is None for parameter in decoder.parameters())

print('Resume checkpoint:', resume_from)
print('Resume epoch:', start_epoch - 1, '-> next epoch:', start_epoch)
print('Best validation BLEU-4:', best_validation_bleu4)
print('Patience:', epochs_without_improvement, '/', PATIENCE)
print('Restored momentum buffers:', momentum_buffers)
print('Finite preflight loss:', float(preflight_loss))
print('GPU allocated/reserved (GB):', round(torch.cuda.memory_allocated()/1024**3, 2),
      round(torch.cuda.memory_reserved()/1024**3, 2))
print('PREFLIGHT PASSED — exact continuation is ready.')

## 6. Recovery-safe training helpers

A new latest checkpoint is written after **every epoch**. Every tenth epoch, a portable ZIP is rebuilt in `/kaggle/working`. The uploaded epoch-95 files remain unchanged. Automatic mixed precision is not introduced because it would alter a run that was trained in full precision through epoch 95.

In [ ]:
BACKUP_ZIP = Path('/kaggle/working/zeromrg_covctr_resume_latest.zip')

def save_history():
    with open(HISTORY_JSON, 'w', encoding='utf-8') as handle:
        json.dump(training_history, handle, indent=2)
    pd.DataFrame(training_history).to_csv(HISTORY_CSV, index=False)

def save_last_checkpoint(epoch):
    temporary = RESUME_LAST.with_suffix('.tmp')
    torch.save({
        'epoch': epoch,
        'decoder': decoder.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': None,
        'best_validation_bleu4': best_validation_bleu4,
        'epochs_without_improvement': epochs_without_improvement,
        'history': training_history,
        'reproduction': {'seed': SEED, 'batch_size': BATCH_SIZE, 'precision': 'float32'},
    }, temporary)
    os.replace(temporary, RESUME_LAST)

def save_best_checkpoint(epoch, validation_bleu4):
    temporary = SELECTED_BEST.with_suffix('.tmp')
    torch.save({
        'epoch': epoch, 'decoder': decoder.state_dict(),
        'validation_bleu4': validation_bleu4,
        'vocabulary_size': len(vocabulary), 'max_tokens': MAX_TOKENS,
    }, temporary)
    os.replace(temporary, SELECTED_BEST)

def create_portable_backup(current_epoch):
    metadata = {
        'current_epoch': current_epoch,
        'best_validation_bleu4': best_validation_bleu4,
        'epochs_without_improvement': epochs_without_improvement,
        'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
        'source_epoch': EXPECTED_SOURCE_EPOCH,
    }
    summary_path = RESULTS_DIR / 'resume_summary.json'
    with open(summary_path, 'w', encoding='utf-8') as handle:
        json.dump(metadata, handle, indent=2)
    archive_items = [
        (RESUME_LAST, 'text_decoder_last.pt'),
        (SELECTED_BEST, 'text_decoder_best.pt'),
        (HISTORY_JSON, 'text_training_history.json'),
        (HISTORY_CSV, HISTORY_CSV.name),
        (summary_path, summary_path.name),
        (TRAINING_LOG, TRAINING_LOG.name),
    ]
    excluded_source_names = {
        'text_decoder_last.pt', 'text_decoder_best.pt', 'text_training_history.json',
        HISTORY_CSV.name, summary_path.name, TRAINING_LOG.name,
    }
    archive_items += [(path, path.name) for path in ARTIFACT_DIR.iterdir()
                      if path.is_file() and path.name not in excluded_source_names]
    temporary = BACKUP_ZIP.with_suffix('.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for path, archive_name in archive_items:
            if path.exists():
                archive.write(path, arcname=archive_name)
    os.replace(temporary, BACKUP_ZIP)
    print(f'Portable backup refreshed at epoch {current_epoch}: {BACKUP_ZIP}')
    print('Backup size (MB):', round(BACKUP_ZIP.stat().st_size / 1024**2, 2))

print('Checkpointing configured: every epoch; portable ZIP every', BACKUP_EVERY, 'epochs.')

## 7. Continue training (one large run, epoch 96 through at most 200)

Run this cell once. It resumes automatically and does not restart from epoch 1. Interrupting the cell loses at most the unfinished epoch because the latest complete epoch is saved. BLEU-4 is the validation selection metric used by this reproduction; test data remain untouched.

In [ ]:
loop_start_epoch = int(training_history[-1]['epoch']) + 1
if loop_start_epoch > MAX_EPOCHS:
    print(f'Nothing to train: checkpoint epoch {loop_start_epoch-1} already reached MAX_EPOCHS={MAX_EPOCHS}.')
elif epochs_without_improvement >= PATIENCE:
    print('Nothing to train: early-stopping patience was already exhausted.')
else:
    print(f'Continuing epochs {loop_start_epoch}..{MAX_EPOCHS}; early stopping patience={PATIENCE}.')
    for epoch in range(loop_start_epoch, MAX_EPOCHS + 1):
        decoder.train()
        epoch_losses = []
        loader = make_loader(train_df, shuffle=True, epoch=epoch)
        for batch in tqdm(loader, desc=f'Text epoch {epoch}', leave=False):
            conditions = feature_batch(train_text_features, batch)
            complete_ids = batch['token_ids'].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = calculate_decoder_loss(conditions, complete_ids)
            if not bool(torch.isfinite(loss)):
                raise RuntimeError(f'Non-finite loss at epoch {epoch}')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), GRADIENT_CLIP)
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))

        mean_loss = float(np.mean(epoch_losses))
        validation_bleu4, references, predictions = validate_text_autoencoder()
        training_history.append({
            'epoch': epoch, 'training_loss': mean_loss,
            'validation_bleu4': validation_bleu4,
        })
        if validation_bleu4 > best_validation_bleu4:
            best_validation_bleu4 = validation_bleu4
            epochs_without_improvement = 0
            save_best_checkpoint(epoch, validation_bleu4)
            improved = True
        else:
            epochs_without_improvement += 1
            improved = False

        save_history()
        save_last_checkpoint(epoch)
        log_line = (
            f'Epoch {epoch:03d} | loss={mean_loss:.4f} | '
            f'validation BLEU-4={validation_bleu4:.4f} | '
            f'best={best_validation_bleu4:.4f} | '
            f'patience={epochs_without_improvement}/{PATIENCE} | improved={improved}'
        )
        print(log_line)
        with open(TRAINING_LOG, 'a', encoding='utf-8') as handle:
            handle.write(log_line + '\n')
        if epoch % BACKUP_EVERY == 0:
            create_portable_backup(epoch)
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            create_portable_backup(epoch)
            break

    final_epoch = training_history[-1]['epoch']
    if not BACKUP_ZIP.exists() or final_epoch % BACKUP_EVERY != 0:
        create_portable_backup(final_epoch)
    print('Training cell finished at epoch:', final_epoch)

## 8. Inspect and export the result

Run this after the training cell finishes or after you deliberately stop it at an epoch boundary. Download the displayed ZIP immediately. Do not evaluate the held-out test set or start zero-shot image inference until early stopping (or the 200-epoch ceiling) has completed and the selected best decoder is fixed.

In [ ]:
history_frame = pd.DataFrame(training_history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_frame['epoch'], history_frame['training_loss'])
axes[0].set(title='Text decoder training loss', xlabel='Epoch', ylabel='Cross-entropy')
axes[0].grid(alpha=0.25)
axes[1].plot(history_frame['epoch'], history_frame['validation_bleu4'])
best_row = history_frame.loc[history_frame['validation_bleu4'].idxmax()]
axes[1].scatter([best_row['epoch']], [best_row['validation_bleu4']], color='red', zorder=3)
axes[1].set(title='Validation BLEU-4', xlabel='Epoch', ylabel='BLEU-4 (%)')
axes[1].grid(alpha=0.25)
fig.tight_layout()
plot_path = RESULTS_DIR / 'text_training_curves.png'
fig.savefig(plot_path, dpi=160, bbox_inches='tight')
plt.show()

last_epoch = int(history_frame.iloc[-1]['epoch'])
if not BACKUP_ZIP.exists():
    create_portable_backup(last_epoch)
print('Last completed epoch:', last_epoch)
print('Best epoch:', int(best_row['epoch']))
print('Best validation BLEU-4:', float(best_row['validation_bleu4']))
print('Latest checkpoint:', RESUME_LAST, '| exists:', RESUME_LAST.exists())
print('Selected best decoder:', SELECTED_BEST, '| exists:', SELECTED_BEST.exists())
print('DOWNLOAD THIS BACKUP:', BACKUP_ZIP)
print('Backup SHA-256:', sha256(BACKUP_ZIP))

## Completion checklist

- The restore cell says source epoch 95 and reports the previous best BLEU-4/patience.
- Preflight says `PREFLIGHT PASSED`, restores momentum buffers, and selects epoch 96 as the first new epoch.
- Training writes a latest checkpoint after every epoch and refreshes the portable ZIP every 10 epochs.
- Stop only when early stopping fires, epoch 200 is reached, or Kaggle time is nearly exhausted.
- Before leaving Kaggle, download `/kaggle/working/zeromrg_covctr_resume_latest.zip` or commit it with Save Version.
- Preserve the console log, history JSON/CSV, training-curve PNG, best checkpoint, and latest checkpoint.
- Share the final epoch, best epoch, best validation BLEU-4, patience value, and ZIP SHA-256 before beginning the zero-shot image-to-report stage.